In [2]:
import sys
import os

# Get the absolute path of the parent directory
parent_dir = os.path.abspath('..')

# Add it to sys.path so Python can find the 'src' module
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Added {parent_dir} to Python path.")

Added /opt/spark to Python path.


**imports and Spark session**

In [3]:
from etl_pipeline.utils.spark_session import get_spark_session
from etl_pipeline.utils.s3_reader import read_delta_table

print("Imports successful")

Imports successful


In [4]:
spark = get_spark_session("SilverLayer-Orders")
print("Spark session ready")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4c89a099-a8a3-4118-a263-dcefd0998f6d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 176ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

Spark session ready


**list Bronze folder using Spark**

In [4]:
from config.settings import S3_BRONZE

# Create URI + Path
uri = spark._jvm.java.net.URI(S3_BRONZE)
path = spark._jvm.org.apache.hadoop.fs.Path(S3_BRONZE)

# Get correct filesystem based on s3a://
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    uri,
    spark._jsc.hadoopConfiguration()
)

# List all items in bronze
files = fs.listStatus(path)

for file in files:
    print(file.getPath().toString())

26/04/06 12:05:17 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


s3a://olist-lakehouse-2026/bronze/holidays_dataset
s3a://olist-lakehouse-2026/bronze/olist_customers_dataset
s3a://olist-lakehouse-2026/bronze/olist_geolocation_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_items_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_payments_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_reviews_dataset
s3a://olist-lakehouse-2026/bronze/olist_orders_dataset
s3a://olist-lakehouse-2026/bronze/olist_products_dataset
s3a://olist-lakehouse-2026/bronze/olist_sellers_dataset
s3a://olist-lakehouse-2026/bronze/product_category_name_translation


# olist_orders_dataset

**read one Bronze dataset**

In [16]:
df_orders = read_delta_table(
    spark,
    "bronze",
    "olist_orders_dataset"
)

print("Bronze dataset loaded successfully")

Bronze dataset loaded successfully


**Preview sample rows**

In [17]:
df_orders.show(10, truncate=False)

[Stage 38:>                                                         (0 + 1) / 1]

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|995392413cee61cc1fcec9807870ea8c|4bf24904ec428325abad2791fcca1aff|delivered   |2017-09-04 22:24:05     |2017-09-04 22:43:54|2017-09-13 17:20:04         |2017-09-22 21:09:32          |2017-09-27 00:00:00          |
|b39de9ed2bb8fd08a970e4517c698824|ed18b557140ff674f36cbcf73921dbbe|delivered   |2018-03-27 09:15:17     |2018-03-27 09:30:12|2018-03-27 23:5

**Inspect schema**

In [8]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



**Count all rows**

In [9]:
total_rows = df_orders.count()
print(f"Total rows: {total_rows}")

Total rows: 99441


**Check distinct business key**

In [10]:
distinct_orders = df_orders.select("order_id").distinct().count()
print(f"Distinct order_id: {distinct_orders}")

[Stage 16:==================================>                       (3 + 2) / 5]

Distinct order_id: 99441


**Quick null profile**

In [11]:
from pyspark.sql.functions import col, when, sum

df_orders.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_orders.columns
]).show()

[Stage 24:==============================================>           (4 + 1) / 5]

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



**inspect order statuses**

In [18]:
df_orders.groupBy("order_status").count().show(truncate=False)

[Stage 41:==============================================>           (4 + 1) / 5]

+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |625  |
|approved    |2    |
|invoiced    |314  |
|delivered   |96478|
|unavailable |609  |
|processing  |301  |
|created     |5    |
+------------+-----+



**save the cleaned dataframe**

In [19]:
df_orders_clean = df_orders

**Save to Silver layer**

In [20]:
from src.utils.s3_writer import write_delta_table

write_delta_table(
    df_orders_clean,
    "silver",
    "olist_orders_dataset_clean"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/olist_orders_dataset_clean


**Important validation after save**

In [7]:
df_silver_test = read_delta_table(
    spark,
    "silver",
    "olist_orders_dataset_clean"
)

print(df_silver_test.count())
df_silver_test.show(5)
df_silver_test.printSchema()

26/04/05 12:15:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

99441


[Stage 13:>                                                         (0 + 1) / 1]

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

# olist_order_items_dataset

**1. Read dataset from Bronze**

In [33]:
df_order_items = read_delta_table(spark, "bronze", "olist_order_items_dataset")
print("Read succesfully")

Read succesfully


**2. Preview some rows**

In [9]:
df_order_items.show(5, truncate=False)

[Stage 11:>                                                         (0 + 1) / 1]

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.9|19.93        |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51032eddd242adc84c38acab88f23d|2018-01-18 14:48:30|199.0|17.87        |
|00024acbcdf0a6daa1e931b038114c75|1            |7634da152a4610f1595efa

**3. Print Schema**

In [11]:
df_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



**4. Check Nulls**

In [12]:
from pyspark.sql.functions import col, count, when

# Summarize null counts for all columns
df_order_items.select([count(when(col(c).isNull(), c)).alias(c) for c in df_order_items.columns]).show()

[Stage 14:===========================================>              (3 + 1) / 4]

+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



**5. Check negative values**

In [15]:
from pyspark.sql import functions as F

# Count negative values in column 'A'
negative_count_price = df_order_items.filter(F.col("price") < 0).count()
print(f"Number of negative values in column price: {negative_count_price}")

negative_count_freight = df_order_items.filter(F.col("freight_value") < 0).count()
print(f"Number of negative values in column freight_value: {negative_count_freight}")

Number of negative values in column price: 0
Number of negative values in column freight_value: 0


**6. Save the cleaned DF then Save to Silver layer**

In [35]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_order_items_cleaned = df_order_items

write_delta_table(
    df_order_items_cleaned,
    "silver",
    "silver_olist_order_items_dataset"
)

print("Save successfully")

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_order_items_dataset
Save successfully


**7. Checking after save**

In [36]:
df_silver_order_items = read_delta_table(
    spark,
    "silver",
    "silver_olist_order_items_dataset"
)

print(df_silver_order_items.count())
df_silver_order_items.show(5)
df_silver_order_items.printSchema()

112650


[Stage 181:>                                                        (0 + 1) / 1]

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|45193dbb3e96a6b68...|            2|a52c53c58fd2105ad...|3b15288545f8928d3...|2017-10-02 16:15:08|106.99|        17.25|
|4519ce49b67354e83...|            1|e09134e776e503444...|6fd52c528dcb38be2...|2018-05-02 02:15:20|129.89|         9.26|
|4519d054994236d26...|            1|e7a7e6b2b4959a39a...|2e90cb1677d35cfe2...|2017-04-18 02:45:21| 18.16|        14.52|
|4519e07ee266cdb76...|            1|9a29b754b7fc0aa8e...|0bf0150d5b9d60d9c...|2017-11-30 17:20:33| 106.0|        20.23|
|4519eb4d4f3192015...|            1|fde71f25e699ca0a2...|bc2ac6b95e1accce9...|2017-08-25 19:25:17|  61.0|         9.34|
+--------------------+-------------+----

# olist_customers_dataset

**1. Read dataset**

In [10]:
df_customers = read_delta_table(
    spark,
    "bronze",
    "olist_customers_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [25]:
df_customers.show(5, truncate=False)
df_customers.printSchema()
print("Total rows:", df_customers.count())

+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|
+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|50edf1bb01f965f14bee7527c3bed42f|58b5f3850e802b91a068ab89ba51686f|27195                   |santanesia   |RJ            |
|91489087a2f86dd3476f0d949946aea2|7b1af6c1d00a207d73873832fc6d18a4|6112                    |osasco       |SP            |
|0d2ff29c36a65d389738620aa2f4e145|8020cb61d67a964fdf5f246d5f6cb201|13418                   |piracicaba   |SP            |
|41e9548a0f8a774ff63837fa26f9839d|92ae32fae1d6d9fcfffd36717ae5166c|5019                    |sao paulo    |SP            |
|2acdf2f6aadbbfede01eccc71529cdb1|9bb28ed7c7fbd7e2392ed20b3fdd9818|96300                   |jaguarao     |RS            |
+-----------------------

**3. Check Nulls**

In [26]:
from pyspark.sql.functions import col, count, when

# Summarize null counts for all columns
df_customers.select([count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns]).show()

[Stage 70:=============================>                            (2 + 2) / 4]

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



**4. Check Distinct Values**

In [34]:
df_customers.select("customer_city").distinct().show()
df_customers.select("customer_state").distinct().show()

+--------------------+
|       customer_city|
+--------------------+
|            camacari|
|           itaberaba|
|           igrejinha|
|                iepe|
|            barracao|
|           arapiraca|
|                pote|
|divino de sao lou...|
|  aguas de sao pedro|
|jijoca de jericoa...|
|              bacaxa|
|   redencao da serra|
|           boa vista|
|            itanhaem|
|                ijui|
|             brusque|
|            perdigao|
|  cachoeira paulista|
|    leandro ferreira|
|           cachoeira|
+--------------------+
only showing top 20 rows



[Stage 127:==========================================>              (3 + 1) / 4]

+--------------+
|customer_state|
+--------------+
|            SC|
|            RO|
|            PI|
|            AM|
|            RR|
|            GO|
|            TO|
|            MT|
|            SP|
|            ES|
|            PB|
|            RS|
|            MS|
|            AL|
|            MG|
|            PA|
|            BA|
|            SE|
|            PE|
|            CE|
+--------------+
only showing top 20 rows



In [49]:
df_customers.select("customer_city").filter(
    col("customer_city").like("sao jose do rio preto")
).show()

[Stage 183:>                                                        (0 + 1) / 1]

+--------------------+
|       customer_city|
+--------------------+
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
+--------------------+
only showing top 20 rows



**5. Save in Silver**

In [11]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_customers_cleaned = df_customers

write_delta_table(
    df_customers_cleaned,
    "silver",
    "silver_olist_customers_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_customers_dataset


**6. Check**

In [12]:
df_silver_customers = read_delta_table(
    spark,
    "silver",
    "silver_olist_customers_dataset"
)

print(df_silver_customers.count())
df_silver_customers.show(5)
df_silver_customers.printSchema()

99443


[Stage 58:>                                                         (0 + 1) / 1]

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|                   68590|             jacunda|            PA|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|                   15056|sao jose do rio p...|            SP|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                   13302|                 itu|            SP|
|7accf3d920f47c07f...|a7f1a6dc9ba06844b...|                   45638|             coaraci|            BA|
|3680a273ddb333253...|6cbfcc29787035834...|                   29700|            colatina|            ES|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows

root
 |-- customer_id: string 

# olist_geolocation_dataset

**1. Read dataset**

In [14]:
df_geolocation = read_delta_table(
    spark,
    "bronze",
    "olist_geolocation_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [15]:
df_geolocation.show(5, truncate=False)
df_geolocation.printSchema()
print("Total rows: ", df_geolocation.count())

+---------------------------+------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat   |geolocation_lng    |geolocation_city|geolocation_state|
+---------------------------+------------------+-------------------+----------------+-----------------+
|50760                      |-8.072588074383969|-34.91359211518264 |recife          |PE               |
|50740                      |-8.042627210658422|-34.946004300305844|recife          |PE               |
|50751                      |-8.066622635581963|-34.917061201223355|recife          |PE               |
|50731                      |-8.038528009435058|-34.93686693228113 |recife          |PE               |
|50720                      |-8.056531245917173|-34.91027403912522 |recife          |PE               |
+---------------------------+------------------+-------------------+----------------+-----------------+
only showing top 5 rows

root
 |-- geolocation_zip_code_prefix: 

**3. Check Nulls**

In [16]:
from pyspark.sql.functions import col, count, when

df_geolocation.select([count(when(col(c).isNull(), c)).alias(c) for c in df_geolocation.columns]).show()

[Stage 75:===================================================>    (11 + 1) / 12]

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



**4. Save in Silver**

In [17]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_geolocation_cleaned = df_geolocation

write_delta_table(
    df_geolocation_cleaned,
    "silver",
    "silver_olist_geolocation_dataset"
)

26/04/05 12:32:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/05 12:32:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/05 12:32:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/05 12:32:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/05 12:32:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/05 12:32:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/05 12:32:28 WARN MemoryManager: Total allocation exceeds 95.0

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_geolocation_dataset


**5. Verify the output**

In [19]:
df_geo_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_geolocation_dataset"
)

df_geo_check.printSchema()
df_geo_check.show(5)
print("Total rows:", df_geo_check.count())

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



+---------------------------+-------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+----------------+-----------------+
|                      74835| -16.73343634893006| -49.26931272652508|         goiania|               GO|
|                      74810|-16.700942999999945| -49.23963799999996|         goiania|               GO|
|                      74893|-16.742034667801384|-49.204687728842764|         goiania|               GO|
|                      74855|-16.724478522679643| -49.22527656263098|         goiania|               GO|
|                      74855|-16.727636999999945|-49.223523499999956|         goiania|               GO|
+---------------------------+-------------------+-------------------+----------------+-----------------+
only showing top 5 rows

Total rows: 1000163


# olist_order_payments_dataset

**1. Read dataset**

In [20]:
df_order_payments = read_delta_table(
    spark,
    "bronze",
    "olist_order_payments_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [21]:
df_order_payments.printSchema()
df_order_payments.show(5, truncate=False)
print("Total rows:", df_order_payments.count())

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|e558bf8bb8fddae947c5ab3a53b8ff64|1                 |credit_card |1                   |371.13       |
|2095778434755b568ba4c3c2325096fe|1                 |credit_card |2                   |55.11        |
|5c1995c020e0a3b2b23152b3fcfcda76|1                 |credit_card |1                   |165.8        |
|ab459ccd2e24ef6d6390517c8b9809a4|1                 |boleto      |1                   |188.62       |
|9b3d13c2644e10e20020ccd7f3d40534|1                 |credit_card |5                   |65.71        |
+--------------------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows

Total rows: 103889


**3. Check Nulls**

In [22]:
from pyspark.sql.functions import col, count, when

df_order_payments.select([count(when(col(c).isNull(), c)).alias(c) for c in df_order_payments.columns]).show()

[Stage 124:======================================>                  (2 + 1) / 3]

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



**4. Distinct values**

In [24]:
df_order_payments.select("payment_type").distinct().show()
print(df_order_payments.count())

+------------+
|payment_type|
+------------+
|      boleto|
| not_defined|
| credit_card|
|     voucher|
|  debit_card|
+------------+

103889


**5. Name friendly**

In [25]:
payment_mapping = {
    "credit_card": "credit card",
    "debit_card": "debit card",
    "not_defined": "not defined"
}

df_order_payments = df_order_payments.replace(
    payment_mapping,
    subset=["payment_type"]
)

In [29]:
df_order_payments.select("payment_type").distinct().show()
print(df_order_payments.count())

+------------+
|payment_type|
+------------+
|  debit card|
|      boleto|
|     voucher|
| credit card|
| not defined|
+------------+

103889


**6. Check negative values**

In [30]:
from pyspark.sql import functions as F

# List of numeric columns to check
cols_to_check = ["payment_sequential", "payment_installments", "payment_value"]

# Create count expressions for each column
count_exprs = [F.count(F.when(F.col(c) < 0, 1)).alias(c) for c in cols_to_check]

df_order_payments.select(count_exprs).show()

[Stage 159:======================================>                  (2 + 1) / 3]

+------------------+--------------------+-------------+
|payment_sequential|payment_installments|payment_value|
+------------------+--------------------+-------------+
|                 0|                   0|            0|
+------------------+--------------------+-------------+



**7. Save in silver**

In [31]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_order_payments_cleaned = df_order_payments

write_delta_table(
    df_order_payments_cleaned,
    "silver",
    "silver_olist_order_payments_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_order_payments_dataset


**8. Verify output**

In [32]:
df_order_payments_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_order_payments_dataset"
)

df_order_payments_check.printSchema()
df_order_payments_check.show(5)
print("Total rows:", df_order_payments_check.count())

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|e558bf8bb8fddae94...|                 1| credit card|                   1|       371.13|
|2095778434755b568...|                 1| credit card|                   2|        55.11|
|5c1995c020e0a3b2b...|                 1| credit card|                   1|        165.8|
|ab459ccd2e24ef6d6...|                 1|      boleto|                   1|       188.62|
|9b3d13c2644e10e20...|                 1| credit card|                   5|        65.71|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows

Total rows: 103889


# olist_order_reviews_dataset

**1. Read dataset**

In [44]:
df_order_reviews = read_delta_table(
    spark,
    "bronze",
    "olist_order_reviews_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [45]:
df_order_reviews.printSchema()
df_order_reviews.show(5, truncate=False)
print("Toal rows: ", df_order_reviews.count())

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                                                                                                |2018-01-18 00:00:00 |2018-01-18 21:46:59    |
|80e641a

**4. Check negative values**

In [35]:
from pyspark.sql import functions as F

# Filter rows where 'col_name' is less than 0
df_order_reviews.filter(F.col("review_score") < 0).show()

[Stage 196:======================================>                  (2 + 1) / 3]

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



**5. Convert data type**

In [46]:
from pyspark.sql.functions import to_date, to_timestamp, col

df_order_reviews = df_order_reviews.withColumn(
    "review_creation_date",
    to_date(col("review_creation_date"))
)

df_order_reviews = df_order_reviews.withColumn(
    "review_answer_timestamp",
    to_timestamp(col("review_answer_timestamp"))
)

df_order_reviews = df_order_reviews.withColumn(
    "review_score",
    col("review_score").cast("int")
)

In [47]:
df_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: date (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [48]:
df_order_reviews.groupBy("review_score").count().show()

[Stage 217:==========================================>              (3 + 1) / 4]

+------------+-----+
|review_score|count|
+------------+-----+
|        NULL| 4935|
|           1|11424|
|           3| 8179|
|           5|57328|
|           4|19142|
|           2| 3151|
|           0|    2|
|          90|    1|
+------------+-----+



**6. Remove outlier**

In [49]:
from pyspark.sql.functions import col

df_order_reviews = df_order_reviews.filter(
    col("review_score").isNull() |
    col("review_score").between(1, 5)
)

In [50]:
df_order_reviews.groupBy("review_score").count().show()

[Stage 222:==============>                                          (1 + 3) / 4]

+------------+-----+
|review_score|count|
+------------+-----+
|        NULL| 4935|
|           1|11424|
|           3| 8179|
|           5|57328|
|           4|19142|
|           2| 3151|
+------------+-----+



**7. Save in silver**

In [51]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_order_reviews_cleaned = df_order_reviews

write_delta_table(
    df_order_reviews_cleaned,
    "silver",
    "silver_olist_order_reviews_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_order_reviews_dataset


**8. Verify output**

In [52]:
df_order_reviews_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_order_reviews_dataset"
)

df_order_reviews_check.printSchema()
df_order_reviews_check.show(5)
print("Total rows:", df_order_reviews_check.count())

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: date (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|e22aad953e113cdcc...|8b2c286fa36b36c81...|           4|                NULL|               Nota 10|          2017-09-19|    2017-09-20 01:07:14|
|5ddb195ab2206a456...|18937b40506fdcbd3...|           4|                NULL|                  NULL|          2017-05-30|    2017-07-06 18:36:02|
|4d3c61768eb47216e...|cb4a79c1e6c9ae443...|           1|                NULL|  Absurdo! Venderam...|          2018-01-05|    2018-01-15 11:04:49|
|5ef9614ed02a28935...|2e1934467537a71d1...|           3|                NULL|                  NULL|          2018-08-01|   

# olist_products_dataset

**1. Read dataset**

In [5]:
df_products = read_delta_table(
    spark,
    "bronze",
    "olist_products_dataset"
)

print("Bronze dataset loaded successfully")

Bronze dataset loaded successfully


**2. Preview dataset**

In [6]:
df_products.printSchema()
df_products.show(5, truncate=False)
print("Total rows: ", df_products.count())

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



26/04/06 02:00:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                 |287                       |1                 |225             |16               |10               |14              |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                 |276                       |1                 |1000            |30               |18               |20              |
|96bd76ec8810374ed1b65e291975717f|e

**3. Check Nulls**

In [7]:
from pyspark.sql.functions import col, count, when

# Counts nulls for every column in the DataFrame
df_products.select([count(when(col(c).isNull(), c)).alias(c) for c in df_products.columns]).show()

[Stage 16:>                                                         (0 + 1) / 1]

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



**4. Check negative values**

In [8]:
from pyspark.sql.functions import col, count, when

numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

df_products.select([
    count(
        when(col(c) < 0, c)
    ).alias(c)
    for c in numeric_cols
]).show()

[Stage 21:>                                                         (0 + 1) / 1]

+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|                  0|                         0|                 0|               0|                0|                0|               0|
+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



**5. Check Distinct values**

In [11]:
df_products.groupBy("product_category_name").count().show()

[Stage 42:>                                                         (0 + 1) / 1]

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
|                  pcs|   30|
|                bebes|  919|
|                artes|   55|
|            cine_foto|   28|
|     moveis_decoracao| 2657|
|             pc_gamer|    3|
| construcao_ferram...|  400|
| tablets_impressao...|    9|
| fashion_roupa_mas...|   95|
|    artigos_de_festas|   26|
|     artigos_de_natal|   65|
|           la_cuisine|   10|
|               flores|   14|
|      livros_tecnicos|  123|
|                 NULL|  610|
|       telefonia_fixa|  116|
| construcao_ferram...|   91|
|           cool_stuff|  789|
|     eletrodomesticos|  370|
|    livros_importados|   31|
+---------------------+-----+
only showing top 20 rows



In [12]:
from pyspark.sql.functions import col, regexp_replace

df_products = df_products.withColumn(
    "product_category_name",
    regexp_replace(col("product_category_name"), "_", " ")
)

In [13]:
df_products.groupBy("product_category_name").count().show()

[Stage 47:>                                                         (0 + 1) / 1]

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
| construcao ferram...|  400|
|                  pcs|   30|
|                bebes|  919|
|                artes|   55|
| fashion bolsas e ...|  849|
|     artigos de natal|   65|
| portateis cozinha...|   10|
| sinalizacao e seg...|   93|
| tablets impressao...|    9|
|               flores|   14|
|    alimentos bebidas|  104|
|         beleza saude| 2444|
| instrumentos musi...|  289|
|                 NULL|  610|
| portateis casa fo...|   31|
| fashion roupa inf...|    5|
|     malas acessorios|  349|
|            cine foto|   28|
|     eletrodomesticos|  370|
| construcao ferram...|   39|
+---------------------+-----+
only showing top 20 rows



**6. Save in silver**

In [14]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_products_cleaned = df_products

write_delta_table(
    df_products_cleaned,
    "silver",
    "silver_olist_products_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_products_dataset


**7. Verify output**

In [15]:
df_products_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_products_dataset"
)

df_products_check.printSchema()
df_products_check.show(5)
print("Total rows:", df_products_check.count())

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte lazer|                 46|                       250|    

# olist_sellers_dataset

**1. Read dataset**

In [16]:
df_sellers = read_delta_table(
    spark,
    "bronze",
    "olist_sellers_dataset"
)

print("success")

success


**2. Preview dataset**

In [17]:
df_sellers.printSchema()
df_sellers.show(5, truncate=False)
print("Total rows:", df_sellers.count())

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



+--------------------------------+----------------------+-----------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|
+--------------------------------+----------------------+-----------------+------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |rio de janeiro   |RJ          |
|c0f3eea2e14555b6faeea3dd58c1b1c3|4195                  |sao paulo        |SP          |
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |braganca paulista|SP          |
+--------------------------------+----------------------+-----------------+------------+
only showing top 5 rows

Total rows: 3095


**3. Check Nulls**

In [18]:
from pyspark.sql.functions import col, count, when

# Counts nulls for every column in the DataFrame
df_sellers.select([count(when(col(c).isNull(), c)).alias(c) for c in df_sellers.columns]).show()

[Stage 83:>                                                         (0 + 1) / 1]

+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



**4. Check distinct**

In [19]:
df_sellers.groupBy("seller_city").count().show()
df_sellers.groupBy("seller_state").count().show()

+--------------------+-----+
|         seller_city|count|
+--------------------+-----+
|           igrejinha|    1|
|             brusque|   10|
|            buritama|    1|
|         carapicuiba|   10|
|               garca|    4|
|  sao joao de meriti|    3|
|    fernando prestes|    1|
|             ipaussu|    1|
|           jacutinga|    3|
|       nova friburgo|    5|
|              araras|    3|
| sao pedro da aldeia|    1|
|              santos|   16|
|itapecerica da serra|    4|
|            ibitinga|   49|
|               muqui|    1|
|              cuiaba|    2|
|     franco da rocha|    2|
|               cotia|   10|
|             marilia|   15|
+--------------------+-----+
only showing top 20 rows



[Stage 93:>                                                         (0 + 1) / 1]

+------------+-----+
|seller_state|count|
+------------+-----+
|          SC|  190|
|          RO|    2|
|          PI|    1|
|          AM|    1|
|          GO|   40|
|          MT|    4|
|          SP| 1849|
|          PB|    6|
|          ES|   23|
|          RS|  129|
|          MS|    5|
|          MG|  244|
|          PA|    1|
|          BA|   19|
|          SE|    2|
|          PE|    9|
|          CE|   13|
|          RN|    5|
|          RJ|  171|
|          MA|    1|
+------------+-----+
only showing top 20 rows



**5. Save in silver**

In [20]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_sellers_cleaned = df_sellers

write_delta_table(
    df_sellers_cleaned,
    "silver",
    "silver_olist_sellers_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_sellers_dataset


**6. Verify output**

In [21]:
df_sellers_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_sellers_dataset"
)

df_sellers_check.printSchema()
df_sellers_check.show(5)
print("Total rows:", df_sellers_check.count())

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
+--------------------+----------------------+-----------------+------------+
only showing top 5 rows

Total rows: 3095


# product_category_name_translation

**1. Read dataset**

In [22]:
df_product_category_name_translation = read_delta_table(
    spark,
    "bronze",
    "product_category_name_translation"
)

print("success")

success


**2. Preview dataset**

In [23]:
df_product_category_name_translation.printSchema()
df_product_category_name_translation.show(5, truncate=False)
print("Total rows:", df_product_category_name_translation.count())

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



+----------------------+-----------------------------+
|product_category_name |product_category_name_english|
+----------------------+-----------------------------+
|beleza_saude          |health_beauty                |
|informatica_acessorios|computers_accessories        |
|automotivo            |auto                         |
|cama_mesa_banho       |bed_bath_table               |
|moveis_decoracao      |furniture_decor              |
+----------------------+-----------------------------+
only showing top 5 rows

Total rows: 71


**3. Check nulls**

In [24]:
from pyspark.sql.functions import col, count, when

# Counts nulls for every column in the DataFrame
df_product_category_name_translation.select([count(when(col(c).isNull(), c)).alias(c) for c in df_product_category_name_translation.columns]).show()

[Stage 129:>                                                        (0 + 1) / 1]

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



**4. Name friendly**

In [25]:
from pyspark.sql.functions import col, regexp_replace

columns_to_clean = [
    "product_category_name",
    "product_category_name_english"
]

for c in columns_to_clean:
    df_product_category_name_translation = df_product_category_name_translation.withColumn(
        c,
        regexp_replace(col(c), "_", " ")
    )

In [26]:
df_product_category_name_translation.groupBy("product_category_name").count().show()
df_product_category_name_translation.groupBy("product_category_name_english").count().show()

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
| construcao ferram...|    1|
|                  pcs|    1|
|                bebes|    1|
|                artes|    1|
| fashion bolsas e ...|    1|
|     artigos de natal|    1|
| sinalizacao e seg...|    1|
| tablets impressao...|    1|
|               flores|    1|
|    alimentos bebidas|    1|
|         beleza saude|    1|
| instrumentos musi...|    1|
| portateis casa fo...|    1|
| fashion roupa inf...|    1|
|     malas acessorios|    1|
|            cine foto|    1|
|     eletrodomesticos|    1|
| construcao ferram...|    1|
|    livros importados|    1|
| fashion roupa mas...|    1|
+---------------------+-----+
only showing top 20 rows



[Stage 139:>                                                        (0 + 1) / 1]

+-----------------------------+-----+
|product_category_name_english|count|
+-----------------------------+-----+
|                          art|    1|
|          musical instruments|    1|
|                      flowers|    1|
|          diapers and hygiene|    1|
|         fashion bags acce...|    1|
|             office furniture|    1|
|                 garden tools|    1|
|              furniture decor|    1|
|                    computers|    1|
|         industry commerce...|    1|
|                         auto|    1|
|         fashion childrens...|    1|
|           christmas supplies|    1|
|         signaling and sec...|    1|
|              home appliances|    1|
|         costruction tools...|    1|
|         fashion underwear...|    1|
|                         food|    1|
|         books general int...|    1|
|         construction tool...|    1|
+-----------------------------+-----+
only showing top 20 rows



**5. Save in silver**

In [27]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_translation_cleaned = df_product_category_name_translation

write_delta_table(
    df_translation_cleaned,
    "silver",
    "silver_product_category_name_translation"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_product_category_name_translation


**6. Verify output**

In [5]:
df_translation_check = read_delta_table(
    spark,
    "silver",
    "silver_product_category_name_translation"
)

df_translation_check.printSchema()
df_translation_check.show(5)
print("Total rows:", df_translation_check.count())

26/04/06 12:17:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



26/04/06 12:17:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza saude|                health beauty|
| informatica acess...|         computers accesso...|
|           automotivo|                         auto|
|      cama mesa banho|               bed bath table|
|     moveis decoracao|              furniture decor|
+---------------------+-----------------------------+
only showing top 5 rows

Total rows: 71


**List silver dataset**

In [6]:
def list_layer_datasets(spark, layer_path):
    uri = spark._jvm.java.net.URI(layer_path)
    path = spark._jvm.org.apache.hadoop.fs.Path(layer_path)

    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        uri,
        spark._jsc.hadoopConfiguration()
    )

    files = fs.listStatus(path)

    return [file.getPath().toString() for file in files]

In [7]:
from config.settings import S3_SILVER

silver_tables = list_layer_datasets(spark, S3_SILVER)

for table in silver_tables:
    print(table)

s3a://olist-lakehouse-2026/silver/silver_olist_customers_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_geolocation_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_items_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_payments_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_reviews_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_orders_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_products_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_sellers_dataset
s3a://olist-lakehouse-2026/silver/silver_product_category_name_translation


In [8]:
spark.stop()